<a href="https://colab.research.google.com/github/YzhangBrian/BookCode/blob/main/Yujie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COM3502-4502-6502 Speech Processing - Python Programming Assignment

## General Information

This programming assignment is worth $55$% of the overall course mark.

You are free to complete this assignment in your own time. However, feedback, advice and guidance are available during the lab classes and via the discussion board on Blackboard.

Note: Via these channels, we try to help you as much as possible, but will not debug your code or provide solutions to the assignment itself.

Note: It will take some time to complete this assignment, so plan your work accordingly over the coming weeks. Read these instructions carefully.

Note: Please be aware that students registered on COM4502 and COM6502 have **additional tasks** to perform. These are marked ‘COM4502-6502 Only’.

Note: You should always ensure that your results (e.g. in terms of plots you create) are clear to understand and leave no room for misinterpretation. This can often be easily achieved by adding proper $x$- and $y$-axis labels, titles, legends etc. Where results are not clear to interpret, this might result in missed points.


## Student Data

Student Family Name: <span style="font-weight:bold;color:orange">**Yujie**</span>

Student Given Name(s): <span style="font-weight:bold;color:orange">**Zhang**</span>

Date of submission: <span style="font-weight:bold;color:orange">**16-12-2025**</span>

## Copyright

This programming assignment is part of the lecture COM[3502](http://www.dcs.shef.ac.uk/intranet/teaching/public/modules/level3/com3502.html "Open web page for COM3502 module")-[4502](http://www.dcs.shef.ac.uk/intranet/teaching/public/modules/level4/com4502.html "Open web page for COM4502 module")-[6502](http://www.dcs.shef.ac.uk/intranet/teaching/public/modules/msc/com6502.html "Open web page for COM4502 module") Speech Processing at the [University of Sheffield](https://www.sheffield.ac.uk/ "Open web page of The University of Sheffield"), [School of Computer Science](https://www.sheffield.ac.uk/cs "Open web page of School of Computer Science"), University of Sheffield.


This notebook is licensed as an assignment to be used during the lecture COM3502-4502-6502 Speech Processing at the University of Sheffield. Any further use is only permitted if agreed with the [module lead](mailto:s.goetze@sheffield.ac.uk).

It should be a matter of course that rules of [unfair means](https://www.sheffield.ac.uk/apse/apo/quality/assessment/unfair) apply and the assignment is not to be shared with or made available to other persons besides those participating in the module during the same academic year. This includes publishing on web pages etc. All questions can be asked during the lab classes or using the Blackboard Discussion board.


## Hand-In Procedure and Deadline

Once you have completed the assignment you should submit a `.zip` file (via Blackboard) containing your solution (as a file named `YourFamilyName.ipynb`) and possibly other sources linked in your Jupyter Notebook. Also, the `.zip` filename should be of the form `YourFamilyName.zip`. Please also ensure that your name is entered correctly in the section above.

Standard school penalties apply for late hand-in and plagiarism.

The **deadline** for handing-in this assignment (via Blackboard) is
<span style="font-weight:bold;color:red">**15:00 on Tuesday, 16th December 2025**</span>.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 0:**
    
Ensure that you data is correctly entered in the section at the top of this sheet and that the filename is in the form `YourFamilyName.ipynb`.
    
</div>

## Libraries

You should be familiar with the use of the following Python libraries from the lab. You should not need to use additional ones. You are allowed to use additional libraries if necessary for your code. If they need to be installed by `!pip install <libraryname>` or `!conda install <libraryname>`, please indicate this as a comment in your code. You should not make use of libraries that can't be installed by either `!pip install` or `!conda install`. You must ensure that your Notebook runs "out of the box". You can test this on the Computer Lab machines in the Diamond if you are unsure and using your own computer.

In [ ]:
#Let's do some necessary and nice-to-have imports
%matplotlib inline
import matplotlib.pyplot as plt    # plotting
#import seaborn as sns; sns.set()  # styling
import numpy as np                 # math

import soundfile as sf             # to load files
from IPython import display as ipd # for sound playback

from scipy import signal           # filter designs (if not already imported)

# Download, load, and analyse audio

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T1:**
    
* Load a wave file containing speech. You can find a file at <a href="https://staffwww.dcs.shef.ac.uk/people/n.ma/comx502/speech.wav">https://staffwww.dcs.shef.ac.uk/people/n.ma/comx502/speech.wav</a> and should be able to download this. You can also use your own WAVE files if you prefer this. If you want to record WAVE files and are using your own computer, the program [Audacity](https://www.audacityteam.org/) is one possibility to [record WAVE files](https://manual.audacityteam.org/man/basic_recording_editing_and_exporting.html).
    
* Visualise the signal in time domain, in the spectral domain (as spectrum) and as a time-frequency representation (spectrogram). Please ensure proper axis labels for all your plot in this assignment.

* Playback the signal.
    
</div>

In [ ]:

# ===== Task T1: load, visualize, and play a speech waveform =====
!curl https://staffwww.dcs.shef.ac.uk/people/S.Goetze/sound/speech.wav -o speech.wav
# 1) Read the audio (convert to mono + float32 in [-1,1])
file_name = "speech.wav"
x, fs = sf.read(file_name, always_2d=False)
print(f"Loaded: {file_name} fs={fs} Hz shape={np.shape(x)} dtype={getattr(x,'dtype',type(x))}")

# stereo -> mono
if x.ndim == 2:
    x = x.mean(axis=1)
    print("Converted to mono:", x.shape)

# integer -> float32 in [-1, 1]
if getattr(x, "dtype", None) is not None and x.dtype.kind in "iu":
    x = x.astype(np.float32) / np.iinfo(x.dtype).max
else:
    x = np.asarray(x, dtype=np.float32)

N = len(x)
duration = N / fs
t = np.arange(N) / fs
print(f"Duration: {duration:.2f} s ({N} samples)")

# 3) Time-domain plot
plt.figure(figsize=(10, 3))
plt.plot(t, x)
plt.title("Time-domain waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

# 4) Amplitude spectrum (one-sided)
# Use next power-of-two FFT length for nicer resolution
def next_pow2(n: int) -> int:
    return 1 if n <= 1 else 1 << (int(np.ceil(np.log2(n))))

Nfft = next_pow2(N)
X = np.fft.rfft(x, n=Nfft)
f = np.fft.rfftfreq(Nfft, d=1.0/fs)
mag = np.abs(X)

plt.figure(figsize=(10, 3))
plt.plot(f, mag)
plt.title("Amplitude spectrum (one-sided)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("|X(f)|")
plt.xlim(0, fs/2)
plt.tight_layout()
plt.show()

# 5) Spectrogram (magnitude in dB)  —— 使用 plt.specgram
win_length_s = 0.025   # 25 ms window
hop_length_s = 0.010   # 10 ms hop
NFFT = int(round(win_length_s * fs))
noverlap = NFFT - int(round(hop_length_s * fs))
if noverlap < 0:
    noverlap = int(0.5 * NFFT)

plt.figure(figsize=(10, 3))
Pxx, freqs, bins, im = plt.specgram(
    x,
    NFFT=NFFT,
    Fs=fs,
    noverlap=noverlap,
    scale='dB'
)
plt.title(f"Spectrogram (Hann, {win_length_s*1e3:.0f} ms window / {hop_length_s*1e3:.0f} ms hop)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.ylim(0, fs/2)
plt.tight_layout()
plt.show()

# 6) Playback
ipd.display(ipd.Audio(x, rate=fs))

audio_in = x
audio_fs = fs

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q1:**

* Determine the sampling frequency $f_s$ in Hz and the length of the signal in seconds. What is the sampling interval $T_s$ of your signal? What is the highest occurring frequency?
    
Note: You can either give your answer in the form of a code block (e.g. by using the `print()` functions) or as text. For the latter, change the [type of the next cell](https://jupyter-notebook.readthedocs.io/en/stable/notebook.html#structure-of-a-notebook-document) from `code` to `markdown` or use the yellow example text below.
    
</div>

In [ ]:
try:
    # We expect variables x (waveform) and fs (sampling rate) from the previous cell (Task T1)
    N = len(x)
    duration = N / fs
    Ts = 1.0 / fs
    f_nyquist = fs / 2.0

    print(f"Sampling frequency fs : {fs} Hz")
    print(f"Number of samples N : {N}")
    print(f"Duration : {duration:.3f} s")
    print(f"Sampling period Ts : {Ts:.6f} s")
    print(f"Highest representable freq : {f_nyquist:.1f} Hz (Nyquist frequency)")
except NameError as e:
    raise RuntimeError("Please run the previous task (Task T1) first to obtain x and fs.") from e


<span style="font-weight:bold;color:orange">In case you want to answer by written text, we would appreciate if you  colour-code your answers, e.g. like using orange font colour as illustrated in this example. This helps us, not to overlook parts of your answers.</span>

## Signal Analysis

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T2:**
    
* Generate a synthetic audio signal consisting of three sinusoids with frequencies of $100$ Hz, $300$ Hz, and $500$ Hz including an initial phase which should be different from zero at least for one of the sinusoids. Assume a sampling rate of $8$ kHz and a duration of $2$ seconds. Write code to generate and plot the waveform of this signal. Compute and plot the magnitude spectrum of the signal.
    
* Use the Fast Fourier Transform (FFT) to calculate the spectrum, and display only the positive frequencies. Identify the main frequency components in the signal (using Python code) based on the magnitude spectrum and briefly explain your observations and identify the three main frequency peaks (as written answer below).

</div>

In [ ]:
# 1) Parameters
fs = 8000          # sampling rate in Hz
dur = 2.0          # duration in seconds
t = np.arange(0, dur, 1/fs)  # time axis (N = fs * dur)

# target sinusoids: amplitudes, frequencies (Hz), and initial phases (rad)
freqs = np.array([100.0, 300.0, 500.0])
amps = np.array([1.0, 0.8, 0.6])
phases = np.array([0.0, np.pi/6, np.pi/3])  # at least one non-zero initial phase

# 2) Synthesize: x(t) = Σ A_k * sin(2π f_k t + φ_k)
x = np.zeros_like(t)
for A, f0, phi in zip(amps, freqs, phases):
    x += A * np.sin(2*np.pi*f0*t + phi)

# 3) Plot waveform
plt.figure(figsize=(10, 3))
plt.plot(t, x)
plt.title("Waveform of 3-sinusoid signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
# Only the first 100 milliseconds
plt.xlim(0, 0.1)
plt.tight_layout()
plt.show()

# 4) One-sided amplitude spectrum via FFT (positive frequencies only)
N = len(x)
X = np.fft.rfft(x)                      # FFT for non-negative frequencies (includes DC and Nyquist)
f = np.fft.rfftfreq(N, d=1/fs)          # frequency axis (Hz)

# Convert to one-sided amplitude spectrum (scale by 2 except DC/Nyquist)
mag = np.abs(X) / N * 2.0
mag[0] /= 2.0
if N % 2 == 0:
    mag[-1] /= 2.0

plt.figure(figsize=(10, 3))
plt.plot(f, mag)
plt.title("One-sided amplitude spectrum")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.xlim(0, 1000)                       # show only 0–1 kHz
plt.tight_layout()
plt.show()

def find_top3_peaks(freqs, mag, min_distance_hz=20.0):
    freqs = np.asarray(freqs)
    mag = np.asarray(mag)

    df = freqs[1] - freqs[0]
    min_dist_bins = int(min_distance_hz / df)

    # Sort the index by magnitude from largest to smallest
    sorted_idx = np.argsort(mag)[::-1]

    chosen = []
    for idx in sorted_idx:
        too_close = False
        for c in chosen:
            if abs(idx - c) < min_dist_bins:
                too_close = True
                break
        if not too_close:
            chosen.append(idx)
        if len(chosen) == 3:
            break

    chosen = np.array(chosen)
    peak_freqs = freqs[chosen]
    peak_mags  = mag[chosen]
    return peak_freqs, peak_mags, chosen

top_freqs, top_amps, top_idx = find_top3_peaks(f, mag, min_distance_hz=20.0)

print("Detected top-3 peaks (Hz, amplitude):")
for fr, am in sorted(zip(top_freqs, top_amps)):
    print(f" {fr:.1f} Hz  |  {am:.3f}")


print("Detected top-3 peaks (Hz, amplitude):")
for fr, am in sorted(zip(top_freqs, top_amps)):
    print(f" {fr:.1f} Hz  |  {am:.3f}")

# optional: annotate peaks on the spectrum
plt.figure(figsize=(10, 3))
plt.plot(f, mag)
for fr, am in zip(top_freqs, top_amps):
    plt.axvline(fr, linestyle="--", alpha=0.6)
    plt.text(fr, am, f"{fr:.0f} Hz", ha="center", va="bottom", rotation=90)
plt.title("Amplitude spectrum with detected peaks")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.xlim(0, 1000)
plt.tight_layout()
plt.show()

x_T2 = x.copy()
fs_T2 = fs



<span style="font-weight:bold;color:orange">The signal consists of three sinusoidal components at 100 Hz, 300 Hz, and 500 Hz.
At least one of these components has a nonzero initial phase.

The sampling rate is 8 kHz, and the duration is 2 s. This gives a frequency
resolution for the FFT of df = fs / N = 8000 / 16000 = 0.5 Hz. All three
frequencies align perfectly with integer frequency bins (200, 600, and 1000).
As a result, there is little spectral leakage, and they show up as three
distinct spectral lines in the amplitude spectrum.

The amplitude spectrum only depends on the amplitudes, not the initial phases.
Therefore, a nonzero initial phase does not affect the peak locations.

Using a simple peak search implemented with NumPy (sorting the spectrum and
enforcing a minimum frequency distance between peaks), we automatically
identify three main peaks at around 100 Hz, 300 Hz, and 500 Hz, which matches
the original construction of the signal.
</span>

# Piece-wise linear filtering in the time domain

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T3:**
    
* Design a high-pass filter with a cut-off frequency of $\approx 500$ Hz using a filter design method of your choice.
* Design a low-pass filter with a cut-off frequency of $\approx 500$ Hz.
* Design a band-stop filter with a cut-off frequencies of $\approx 300$ Hz and of $\approx 1.1$ kHz.
* Simulate the effect of a land-line telephone by eliminating all energy below $300$ Hz and above $3,400$ Hz.
* Visualise the transfer functions of the filters and the zero-pole plots.
* Apply the designed filters, compare filter input and output as a time-frequency visualisation and play back the filtered signal.
    
Note: Don't forget proper labeling/description of your figures to make clear what is what.
    
Note: In case you encounter stability problems, remember that we mentioned in the lecture, that filters can be designed as second-order-systems (SOS) which the design methods you are familiar with can realise.
</div>

In [ ]:
# ===== Task T3: filter design, z-plane, spectrograms & playback =====

from scipy import signal as sig

# Prepare input signal x, fs
try:
    x, fs
except NameError:

    x, fs = sf.read("speech.wav", always_2d=False)

# stereo -> mono
if x.ndim == 2:
    x = x.mean(axis=1)

x = np.asarray(x, dtype=np.float32)

def design_butter(filter_type, cutoff_hz, order, fs):
    """
    Butterworth filter design.
    filter_type: 'lowpass', 'highpass', 'bandpass', 'bandstop'
    cutoff_hz  : Single frequency or (low, high), in Hz
    """
    nyq = 0.5 * fs
    if np.isscalar(cutoff_hz):
        Wn = cutoff_hz / nyq
    else:
        Wn = [c / nyq for c in cutoff_hz]

    btype_map = {
        "lowpass":  "low",
        "highpass": "high",
        "bandpass": "bandpass",
        "bandstop": "bandstop",
    }
    btype = btype_map[filter_type]

    b, a = sig.butter(order, Wn, btype=btype)
    return b, a


def plot_response(b, a, fs, title):
    """Plot the amplitude response (dB) using freqz。"""
    w, h = sig.freqz(b, a, worN=1024)
    freqs = w * fs / (2.0 * np.pi)
    mag_db = 20 * np.log10(np.maximum(np.abs(h), 1e-12))

    plt.figure(figsize=(8, 3))
    plt.plot(freqs, mag_db)
    plt.axhline(-3, ls="--", alpha=0.6, label="-3 dB")
    plt.title(f"{title} — Magnitude response")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude (dB)")
    plt.xlim(0, fs/2)
    plt.grid(True, ls=":")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()


def apply_filter(b, a, sig_in):
    """Apply an IIR filter using filtfilt (zero phase, consistent with lab)."""
    sig_in = np.asarray(sig_in, dtype=float)
    y = sig.filtfilt(b, a, sig_in)
    return y.astype(np.float32)


def zplane_plot(b, a, title="Poles and zeros"):
    z = np.roots(b)
    p = np.roots(a)

    plt.figure(figsize=(4.2, 4.2))

    # Unit circle
    theta = np.linspace(0, 2*np.pi, 512)
    plt.plot(np.cos(theta), np.sin(theta), "k:", lw=1, label="Unit circle")

    if len(z):
        plt.scatter(np.real(z), np.imag(z), marker="o", facecolors="none", edgecolors="b", label="Zeros")
    if len(p):
        plt.scatter(np.real(p), np.imag(p), marker="x", color="r", label="Poles")

    plt.title(title)
    plt.xlabel("Real")
    plt.ylabel("Imag")
    plt.axis("equal")
    plt.grid(True, ls=":")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()


def show_spec_pair(x1, x2, fs, title1, title2, fmax=None):
    """Input/Output Spectrogram Comparison."""
    LDFT = 1024
    if fmax is None:
        fmax = fs / 2

    plt.figure(figsize=(11, 4.2))

    plt.subplot(1, 2, 1)
    plt.specgram(x1, Fs=fs, NFFT=LDFT)
    plt.title(title1)
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.ylim(0, fmax)

    plt.subplot(1, 2, 2)
    plt.specgram(x2, Fs=fs, NFFT=LDFT)
    plt.title(title2)
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.ylim(0, fmax)

    plt.tight_layout()
    plt.show()

order   = 6
hp_fc   = 500.0           # high-pass cutoff ≈ 500 Hz
lp_fc   = 500.0           # low-pass cutoff ≈ 500 Hz
bs_band = (300.0, 1100.0)       # band-stop ≈ 300–1100 Hz
tel_band= (300.0, 3400.0)       # telephone band-pass 300–3400 Hz

b_hp,  a_hp = design_butter("highpass", hp_fc, order, fs)
b_lp,  a_lp = design_butter("lowpass",  lp_fc, order, fs)
b_bs,  a_bs = design_butter("bandstop", bs_band, order, fs)
b_tel, a_tel = design_butter("bandpass", tel_band, order, fs)

# Frequency Response & Zero-Pole Diagram
plot_response(b_hp, a_hp, fs, f"High-pass @ {hp_fc:.0f} Hz")
zplane_plot(b_hp, a_hp, f"High-pass @ {hp_fc:.0f} Hz")

plot_response(b_lp, a_lp, fs, f"Low-pass @ {lp_fc:.0f} Hz")
zplane_plot(b_lp, a_lp,  f"Low-pass @ {lp_fc:.0f} Hz")

plot_response(b_bs, a_bs,  fs, f"Band-stop @ {bs_band[0]:.0f}–{bs_band[1]:.0f} Hz")
zplane_plot(b_bs, a_bs,  f"Band-stop @ {bs_band[0]:.0f}–{bs_band[1]:.0f} Hz")

plot_response(b_tel, a_tel, fs, f"Telephone band-pass @ {tel_band[0]:.0f}–{tel_band[1]:.0f} Hz")
zplane_plot(b_tel, a_tel, f"Telephone band-pass @ {tel_band[0]:.0f}–{tel_band[1]:.0f} Hz")

# For speech application filters
y_hp  = apply_filter(b_hp, a_hp, x)
y_lp  = apply_filter(b_lp, a_lp, x)
y_bs  = apply_filter(b_bs, a_bs, x)
y_tel = apply_filter(b_tel, a_tel, x)

# Spectrogram Comparison & Playback
fmax = min(4000, fs/2)

show_spec_pair(x, y_hp, fs, "Input (original)", f"Output: High-pass @ {hp_fc:.0f} Hz", fmax=fmax)
show_spec_pair(x, y_lp, fs, "Input (original)", f"Output: Low-pass @ {lp_fc:.0f} Hz", fmax=fmax)
show_spec_pair(x, y_bs, fs, "Input (original)", f"Output: Band-stop @ {bs_band[0]:.0f}-{bs_band[1]:.0f} Hz", fmax=fmax)
show_spec_pair(x, y_tel, fs, "Input (original)", f"Output: Telephone {tel_band[0]:.0f}-{tel_band[1]:.0f} Hz", fmax=fmax)

print("Original:"); ipd.display(ipd.Audio(x, rate=fs))
print(f"High-pass {hp_fc:.0f} Hz:"); ipd.display(ipd.Audio(y_hp, rate=fs))
print(f"Low-pass {lp_fc:.0f} Hz:"); ipd.display(ipd.Audio(y_lp, rate=fs))
print(f"Band-stop {bs_band[0]:.0f}-{bs_band[1]:.0f} Hz:"); ipd.display(ipd.Audio(y_bs, rate=fs))
print(f"Telephone {tel_band[0]:.0f}-{tel_band[1]:.0f} Hz:"); ipd.display(ipd.Audio(y_tel, rate=fs))


<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q2:**

* Explain the behaviour of the designed band-stop filter, i.e. describe (briefly) what you can see in the generated plots. If you didn't generate plots you can explain what you would expect to see.

    
</div>

**Answer to Question Q2:**

<span style="font-weight:bold;color:orange">
    ... Behaviour: The band-stop (notch) filter effectively reduces energy between approximately 300 Hz and 1.1 kHz. It does not significantly affect frequencies below 300 Hz or above 1.1 kHz, which remain largely unchanged at about 0 dB. In the magnitude response, you can see a deep attenuation valley in the stopband, with flat passbands on both sides. The zero-pole plot shows zeros near the unit circle in the stopband region, creating the notch, while poles are positioned to control the transition slopes and maintain passband flatness.                                  
    In the spectrogram, energy between 300 and 1100 Hz is suppressed. For speech, this particularly reduces parts of the first formant (F1, typically around 300 to 900 Hz) and some low-frequency harmonics. As a result, vowels lose much of their fullness, while higher-frequency consonant noise stays intact. ...
</span>

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q3:**
    
* Which sounds are most affected when the low-pass cut-off frequency is set to around $500$
Hz - vowels or consonants - and why?
    
</div>

**Answer to Question Q3:**

<span style="font-weight:bold;color:orange">
    ...Most impacted are consonants.  Many consonants require energy well above 1–2 kHz, particularly plosive bursts and unvoiced fricatives (/s/, /ʃ/, /f/).  Most of that high-frequency content is eliminated by a low-pass at about 500 Hz, which renders consonants significantly less understandable or even unintelligible.
 Vowels are still audible because a lot of their energy (F0 and F1 around 300–900 Hz) is below 1 kHz. As higher formants (F2, F3, …) are attenuated, they will sound muffled, but they are not completely removed like high-frequency consonant cues. ...
</span>

# Audio Effects

## Low-Frequency Oscillator

Many ‘voice effects (FXs)’ are achieved by modifying some characteristic of the speech using a low-frequency oscillator or *LFO*. LFOs typically have two controls: speed (which is specified by the frequency in Hertz) and depth (which specifies the magnitude of the effect). The following tasks will require several LFOs, so it makes sense to implement one in the following.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T4:**
    
* Implement a Low Frequency Oscillator as a function `lfo()` as described below. Visualise that your function works by generating a sine and a square wave of frequency $5$ Hz and length $2$ seconds with different depths.
    
Note: There will be an extra point in the marking if you **do not** use the `scipy` library to solve this task.
    
</div>

In [ ]:
def lfo(speed_hz, depth, num_samples, fs=44100, square_curve=False):
    '''
    Low-frequency oscillator

    Parameters
    ----------
    speed_hz : float
       frequency of generated signal in Hertz
    depth : float
        magnitude of the effect
    num_samples : int
        length of the signal in samples
    fs : float, optional
        sampling frequency in Hz, default 44100
    square_curve : boolean, optional (default: False)
        generate square wave if true, generate sine wave if false

    Example use:
    -------
        sig_square = lfo(speed_hz=5, depth=0.7, num_samples=88200, fs=44100, square_curve=True)
    '''

    # Basic validation (lightweight)
    if num_samples <= 0:
        return np.zeros(0, dtype=np.float32)
    if fs <= 0 or speed_hz < 0:
        raise ValueError("fs must be > 0 and speed_hz must be >= 0.")
    if depth < 0:
        raise ValueError("depth must be >= 0.")

    n = np.arange(num_samples, dtype=np.float64)    # sample indices
    phi = 2.0 * np.pi * speed_hz * n / fs           # instantaneous phase

    if square_curve:
        # Square wave in {-1, +1} scaled by 'depth'
        base = np.sin(phi)
        # Avoid exact zeros mapping to 0: force +/-1 for a clean square
        base = np.where(base >= 0.0, 1.0, -1.0)
        y = depth * base
    else:
        # Sine wave in [-1, +1] scaled by 'depth'
        y = depth * np.sin(phi)

    return y.astype(np.float32)

In [ ]:
# Demo parameters
fs = 44100
f0 = 5.0        # LFO speed in Hz
dur = 2.0        # seconds
N = int(dur * fs)

# Generate LFOs with different depths
lfo_sine_d03 = lfo(speed_hz=f0, depth=0.3, num_samples=N, fs=fs, square_curve=False)
lfo_sine_d08 = lfo(speed_hz=f0, depth=0.8, num_samples=N, fs=fs, square_curve=False)
lfo_square_d03 = lfo(speed_hz=f0, depth=0.3, num_samples=N, fs=fs, square_curve=True)
lfo_square_d08 = lfo(speed_hz=f0, depth=0.8, num_samples=N, fs=fs, square_curve=True)

# Time axis for plotting
t = np.arange(N) / fs

# Optional: quick numerical check (peak approx equals 'depth')
print("Sine depth check:", np.max(np.abs(lfo_sine_d03)), np.max(np.abs(lfo_sine_d08)))
print("Square depth check:", np.max(np.abs(lfo_square_d03)), np.max(np.abs(lfo_square_d08)))

# Visualisation
plt.figure(figsize=(10, 6))

# Sine LFOs
plt.subplot(2, 1, 1)
plt.plot(t, lfo_sine_d03, label="sine, depth=0.3")
plt.plot(t, lfo_sine_d08, label="sine, depth=0.8", alpha=0.85)
plt.title("LFO outputs (sine, 5 Hz, 2 s)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(loc="upper right")

# Square LFOs
plt.subplot(2, 1, 2)
plt.plot(t, lfo_square_d03, label="square, depth=0.3")
plt.plot(t, lfo_square_d08, label="square, depth=0.8", alpha=0.85)
plt.title("LFO outputs (square, 5 Hz, 2 s)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(loc="upper right")

plt.tight_layout()
plt.show()

Although your function outputs audio, you are unlikely to be able to hear it as the frequency is so low. However, you can check that it is functioning correctly by combining it with other audio signals as we will do in the following.

## Amplitude Modulation - Tremolo

*Tremolo* is one of the most basic voice manipulations that makes use of an LFO. In this effect, the amplitude of a speech signal is [modulated](https://en.wikipedia.org/wiki/Amplitude_modulation), i.e. the speech waveform is multiplied by a variable gain that ranges between $1-$ `modulation_depth` and $1$. This means that if the `modulation_depth` equals $1$, the variable gain varies between $1-1=0$ and $1$.

Your LFO outputs an audio signal between `-depth` and `+depth` which is different from the `modulation_depth` above. So, in order to modulate the amplitude of the speech correctly, the output of the LFO has to be scaled and applied appropriately.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T5:**
    
* Implement  a function `tremolo()` using your function `lfo()` and modulate the amplitude of the speech signal.
* Experiment with different settings for `speed` and `modulation_depth`. In particular, note that a square wave with a *speed* between $3$ and $4$ Hz (and `modulation_depth` = $1$) has a very destructive effect on the intelligibility of the output. This is because $3-4$ Hz corresponds to the typical syllabic rate of speech.
* Proof that the effect works by a proper visualisation of the filtered speech signal and describe what can be observed and perceived.
    
</div>

In [ ]:
def tremolo(signal, fs, speed_hz, modulation_depth, square_curve=False):
    '''
    Applies a tremolo effect to a signal

    Parameters
    ----------
    signal : float
       input signal to which the effect should be applied
    fs : int
        sampling frequency in Hz
    speed_hz : float
       frequency of LFO and by this also the effect
    modulation_depth : float
        magnitude of the effect
    square_curve : boolean, optional
        generate square wave if true, generate sine wave if false

    Return
    ----------
    signal after application of tremolo effect

    Example use:
    -------
        signal_tremolo = tremolo(signal=audio_in, fs=fs, speed_hz=10, modulation_depth=1, square_curve=False)
    '''

    x = np.asarray(signal, dtype=float)
    N = x.shape[0]
    d = float(np.clip(modulation_depth, 0.0, 1.0))

    l = lfo(speed_hz=speed_hz, depth=d, num_samples=N, fs=fs, square_curve=square_curve)  # ∈ [-d, +d]

    gain = 1.0 - d/2.0 + l/2.0

    y = x * gain
    return y.astype(np.float32)

In [ ]:
# Your code here to show an example of the Tremolo effect
#
if 'audio_in' in globals() and 'fs' in globals():
    x = np.asarray(audio_in, dtype=float)
else:
    fs = 16000
    dur = 2.0
    t = np.arange(int(dur * fs)) / fs
    x = 0.6*np.sin(2*np.pi*220*t) + 0.3*np.sin(2*np.pi*440*t)

y_sine = tremolo(x, fs, speed_hz=8.0, modulation_depth=0.7, square_curve=False)
y_square = tremolo(x, fs, speed_hz=3.5, modulation_depth=1.0, square_curve=True)

sec = 0.4
n = int(sec * fs)
t_short = np.arange(n) / fs

plt.figure(figsize=(10, 3.2))
plt.title('Original vs Tremolo (sine LFO: 8 Hz, depth=0.7)')
plt.plot(t_short, x[:n], label='original')
plt.plot(t_short, y_sine[:n], label='tremolo (sine)')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

plt.figure(figsize=(10, 3.2))
plt.title('Original vs Tremolo (square LFO: 3.5 Hz, depth=1.0)')
plt.plot(t_short, x[:n], label='original')
plt.plot(t_short, y_square[:n], label='tremolo (square)')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

ipd.display(ipd.Audio(x, rate=fs))
ipd.display(ipd.Audio(y_sine, rate=fs))
ipd.display(ipd.Audio(y_square,rate=fs))


## Ring Modulation

Another basic effect is to multiply the speech signal by the output of an LFO. This is known as ‘ring modulation’.

Note: In the BBC TV series [Dr. Who](https://en.wikipedia.org/wiki/Doctor_Who), the voices of the alien [Daleks](https://en.wikipedia.org/wiki/Dalek) are generated by a ring modulator with an LFO set to around 30 Hz. The voice actors also spoke using a stilted monotonic intonation in order to enhance the effect. You can try this yourself by recording your own voice and applying the effect.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T6 (Ring Modulation):**
    
* Implement a function `ring_modulation()` using your function `lfo()` and modulate the amplitude of the speech signal by multiplying with the LFO signal.
* Experiment with different settings for `speed` and `depth`. Note how the timbre of the resulting sound is subtly different from *tremolo*.
* Proof that the effect works by a proper visualisation of the filtered speech signal and describe what can be observed and perceived.
    
</div>

In [ ]:
def ring_modulation(signal, fs, speed_hz, depth, square_curve=False):
    '''
    Applies a ring modulation effect to a signal

    Parameters
    ----------
    signal : float
       input signal to which the effect should be applied
    fs : int
        sampling frequency in Hz
    speed_hz : float
       frequency of LFO and by this also the effect
    depth : float
        magnitude of the effect
    square_curve : boolean, optional
        generate square wave if true, generate sine wave if false

    Return
    ----------
    signal after application of the ring modulation effect

    Example use:
    -------
        signal_ring_mod = ring_modulation(audio_in, fs, 10, 1, square_curve=False)
    '''
#
# ...
    x = np.asarray(signal, dtype=float)
    N = x.shape[0]
    d = float(np.clip(depth, 0.0, 1.0))

    m = lfo(speed_hz=speed_hz,
            depth=d,
            num_samples=N,
            fs=fs,
            square_curve=square_curve)

    if x.ndim > 1:
        m = m[:, None]

    y = x * m
    return y.astype(np.float32)

In [ ]:
# Your code here to show an example of the Ring Modulation effect
#
# ...
try:
    lfo; ring_modulation
except NameError as e:
    raise NameError("Please run the cells that define lfo() and ring_modulation() before this demo.") from e

fs = int(globals().get('fs', 16000))
audio_g = globals().get('audio_in', None)

if audio_g is not None:
    x = np.asarray(audio_g, dtype=float)
    if x.ndim > 1:
        x = x.mean(axis=1)
else:
    dur = 2.0
    t = np.arange(int(dur * fs)) / fs
    x = 0.6*np.sin(2*np.pi*220*t) + 0.3*np.sin(2*np.pi*440*t)

# Circular Modulation
y_rm_30 = ring_modulation(x, fs, speed_hz=30.0, depth=1.0, square_curve=False)
y_rm_30_sq = ring_modulation(x, fs, speed_hz=30.0, depth=1.0, square_curve=True)

# Drawing + Trial Listening
sec = 0.15
n = int(sec * fs)
t_short = np.arange(n) / fs

m_sine = lfo(speed_hz=30.0, depth=1.0, num_samples=len(x), fs=fs, square_curve=False)
m_sq = lfo(speed_hz=30.0, depth=1.0, num_samples=len(x), fs=fs, square_curve=True)

plt.figure(figsize=(10, 3))
plt.title('Ring Mod (30 Hz sine): original vs output')
plt.plot(t_short, x[:n], label='original')
plt.plot(t_short, y_rm_30[:n], label='ring-mod (30 Hz sine)')
plt.xlabel('Time [s]'); plt.ylabel('Amplitude'); plt.legend(); plt.show()

plt.figure(figsize=(10, 3))
plt.title('Modulators m(t) for ring modulation')
plt.plot(t_short, m_sine[:n], label='m(t) sine 30 Hz')
plt.plot(t_short, m_sq[:n], label='m(t) square 30 Hz', alpha=0.8)
plt.xlabel('Time [s]'); plt.ylabel('m(t)'); plt.legend(); plt.show()

ipd.display(ipd.Audio(x, rate=fs))
ipd.display(ipd.Audio(y_rm_30, rate=fs))
ipd.display(ipd.Audio(y_rm_30_sq, rate=fs))

## Frequency Shifting

Many Vocal FX are the result of altering the frequencies present, e.g. changing the pitch of a voice. There are many algorithms for frequency shifting. You have already implemented an approximate solution with your ring modulator.

For simplicity, the following function will be given implementing frequency shifting.

In [ ]:
# the following code in this cell is taken and slightly adapted from:
# https://gist.github.com/lebedov/4428122

import scipy.signal as sig

def nextpow2(n):
    '''Return the first integer N such that 2**N >= abs(n)'''
    return int(np.ceil(np.log2(np.abs(n))))

def frequency_shift(signal, fs, shift_amount):
    '''
    Shift the specified signal by the specified frequency.

    Parameters
    ----------
    signal : float
       input signal to which the effect should be applied
    fs : int
        sampling frequency in Hz
    shift_amount : float
       amount of frequency shift (in Hz)

    Return
    ----------
    signal after application of the frequency shifting effect

    Example use:
    -------
        signal_frequency_shifted = frequency_shift(audio_in, fs, 100)
    '''

    # Pad the signal with zeros to prevent the FFT invoked by the transform from
    # slowing down the computation:
    N_orig = len(signal)
    N_padded = 2 ** nextpow2(N_orig)
    t = np.arange(0, N_padded)
    return (
        sig.hilbert(
            np.hstack((signal, np.zeros(N_padded - N_orig, signal.dtype)))
        )
        * np.exp(2j * np.pi * shift_amount * t / fs)
    )[:N_orig].real

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T7 (Frequency Shifting):**
    
* Visualise the effect of the frequency shift effect using an appropriate spectral representation.
    
</div>

In [ ]:
# Your code here to show an example of the frequency shift effect
#
# ...

audio_g = globals().get('audio_in', None)
fs_g = globals().get('fs', None)

if (audio_g is not None) and (fs_g is not None):
    x = np.asarray(audio_g, dtype=float)
    Fs = int(fs_g)
    if x.ndim > 1:
        x = x.mean(axis=1)
else:
    Fs = 16000
    dur = 2.0
    t = np.arange(int(dur * Fs)) / Fs
    x = 0.7*np.sin(2*np.pi*220*t) + 0.3*np.sin(2*np.pi*440*t)

shift_up = +800.0   # Hz
shift_down = -800.0   # Hz

y_up = frequency_shift(x, Fs, shift_up)
y_down = frequency_shift(x, Fs, shift_down)

# 3) Amplitude Spectrum (Unilateral Spectrum) Comparison
def mag_spectrum(sig_t, fs):
    N = len(sig_t)
    win = np.hanning(N)
    X = np.fft.rfft(sig_t * win)
    f = np.fft.rfftfreq(N, d=1.0/fs)
    mag = 20*np.log10(np.maximum(np.abs(X), 1e-12))
    return f, mag

sec = 1.0
Nsp = min(len(x), int(sec * Fs))
f0, X0 = mag_spectrum(x[:Nsp], Fs)
f1, Xup = mag_spectrum(y_up[:Nsp], Fs)
f2, Xdn = mag_spectrum(y_down[:Nsp], Fs)

plt.figure(figsize=(10, 4))
plt.title('Magnitude spectrum: original vs frequency-shifted (+800 / -800 Hz)')
plt.plot(f0, X0, label='original')
plt.plot(f1, Xup, label=f'up-shift {shift_up:.0f} Hz')
plt.plot(f2, Xdn, label=f'down-shift {abs(shift_down):.0f} Hz')
plt.xlim(0, Fs/2)
plt.xlabel('Frequency [Hz]'); plt.ylabel('Magnitude [dB]')
plt.legend(); plt.grid(True, alpha=0.25)
plt.show()

# 4) Spectrogram
def show_spectrogram(sig_t, fs, title):
    plt.figure(figsize=(9, 3))
    nfft = 1024; hop = nfft // 4
    plt.specgram(sig_t, NFFT=nfft, Fs=fs, noverlap=nfft-hop, cmap='magma')
    plt.title(title); plt.xlabel('Time [s]'); plt.ylabel('Frequency [Hz]')
    plt.ylim(0, fs/2); plt.colorbar(label='Power [dB]')
    plt.show()

show_spectrogram(x, Fs, 'Spectrogram: original')
show_spectrogram(y_up, Fs, f'Spectrogram: up-shift {shift_up:.0f} Hz')
show_spectrogram(y_down, Fs, f'Spectrogram: down-shift {abs(shift_down):.0f} Hz')

ipd.display(ipd.Audio(x, rate=Fs))
ipd.display(ipd.Audio(y_up, rate=Fs))
ipd.display(ipd.Audio(y_down, rate=Fs))

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q4:**

* COM3502-4502-6502: Why can the voice be shifted up in frequency much further than
it can be shifted down in frequency before it becomes severely distorted? Hint: Calculate a spectrum plot if the answer is not immediately clear to you.
* COM4502-6502 ONLY: Your frequency shifter changes all the frequencies present in an input signal. How might it be possible to change the pitch of a voice without altering the formant frequencies?
    
</div>

**Answer to Question Q4:**

<span style="font-weight:bold;color:orange">
    ...

When you shift down, those low components are pushed to/ below 0 Hz → they fold/overlap when we return to a real signal (and are often cut by high‑pass filters). Result: strong distortion after only a small downward shift.

When you shift up, the whole spectrum moves away from 0 Hz; nothing collides until you approach Nyquist (fs/2), so you can shift much further up without severe distortion.

downward headroom is small because the voice touches 0 Hz; upward headroom is large until you reach Nyquist....
</span>

## Harmony Effect

A classic ‘robotic’ voice can be achieved by simply adding frequency-shifted speech back to the unprocessed original. This effect is known as ‘harmony’. However, rather than simply adding the signals in equal amounts, we will implement a more general-purpose approach.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T8:**
    
* Implement a function `mixer()` that adds the original speech with the manipulated speech in different proportions.
* Implement a function `harmony()` that mixes the input signal with a frequency-shifted version of itself (using the functions `mixer()` and `frequency_shift()`). With your mixer at the $50$-$50$ setting, experiment with different frequency shifts in order to produce the best robotic-sounding output. Report "your optimal" setting.
</div>

In [ ]:
def mixer(signal1, signal2, percentage_1=0.5):

 # Your code here to implement mixing of two signals, used later for the harmony effect
 x1 = np.asarray(signal1, dtype=float).squeeze()
 x2 = np.asarray(signal2, dtype=float).squeeze()

 # Alignment Length
 N = min(len(x1), len(x2))
 if N == 0:
  return np.zeros(0, dtype=np.float32)

 p = float(np.clip(percentage_1, 0.0, 1.0))
 y = p * x1[:N] + (1.0 - p) * x2[:N]

 m = np.max(np.abs(y))
 if m > 1.0 and m > 0:
   y = y / m

 return y.astype(np.float32)

In [ ]:
def harmony(signal, fs, shift_hz, percentage_1=0.5):
 shifted = frequency_shift(signal, fs, shift_hz)
 return mixer(signal, shifted, percentage_1=percentage_1)

In [ ]:
audio_g = globals().get('audio_in', None)
fs_g = globals().get('fs', None)

if (audio_g is not None) and (fs_g is not None):
    x = np.asarray(audio_g, dtype=float)
    fs = int(fs_g)
    if x.ndim > 1:
        x = x.mean(axis=1)
else:
    fs = 16000
    dur = 2.0
    t = np.arange(int(dur * fs)) / fs
    x = 0.7*np.sin(2*np.pi*220*t) + 0.3*np.sin(2*np.pi*440*t)

shift_demo = +60.0
y_harm = harmony(x, fs, shift_hz=shift_demo, percentage_1=0.5)

n_show = int(0.10 * fs)
t_short = np.arange(n_show) / fs
plt.figure(figsize=(10, 3.2))
plt.title(f'Harmony example (mix=50/50, shift={shift_demo:.0f} Hz)')
plt.plot(t_short, x[:n_show], label='original')
plt.plot(t_short, y_harm[:n_show], label='harmony')
plt.xlabel('Time [s]'); plt.ylabel('Amplitude'); plt.legend(); plt.show()

ipd.display(ipd.Audio(x, rate=fs))
ipd.display(ipd.Audio(y_harm, rate=fs))

**Answer to question in Task T7:**

<span style="font-weight:bold;color:orange">
    ...your answer here ... <br>
  My optimal setting is: 50/50 mix with a +60 Hz frequency shift. <br>
    Because: mixing the original with a +60 Hz–shifted copy creates an amplitude beating of about 30 Hz (Δf/2), which gives a strong “robotic”/“Dalek-like” character while keeping speech intelligible. Smaller shifts sound like chorus; much larger shifts become too buzzy.
</span>

## Frequency Modulation: Vibrato

Now that you have the ability to shift the frequencies in a speech signal, it is very easy to implement another common voice manipulation technique - *vibrato*. All that is required is for the frequency shifter to be controlled by the output of an LFO.



<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T9:**
    
* Implement a function `vibrato()` by connecting an LFO to your frequency shifter, and experiment with different values for speed and depth. Note that the LFO output will need to be scaled to provide an appropriate frequency shift range and then added to the output of the frequency shift.
    
</div>



In [ ]:
def vibrato(signal, fs, speed_hz, shift_hz, square_curve=False):

    # Convert input to 1-D float array
    x = np.asarray(signal, dtype=float)
    if x.ndim > 1:
        # simple mono mix if multi-channel
        x = x.mean(axis=1)
    N = x.shape[0]

    if N == 0:
        return np.zeros(0, dtype=np.float32)

    # Pre-compute two frequency-shifted versions:
    # one shifted up by +shift_hz, one down by -shift_hz.
    # Uses the frequency_shift() function defined earlier.
    y_up = frequency_shift(x, fs, +shift_hz)
    y_down = frequency_shift(x, fs, -shift_hz)

    # LFO in range [-1, +1] (depth=1.0)
    m = lfo(speed_hz=speed_hz,
            depth=1.0,
            num_samples=N,
            fs=fs,
            square_curve=square_curve)

    # Convert LFO output to crossfade weights between y_down and y_up:
    # m = -1 -> all "down"; m = +1 -> all "up"
    w_up = 0.5 * (1.0 + m)     # in [0, 1]
    w_down = 1.0 - w_up        # also in [0, 1], w_up + w_down = 1

    # Apply time-varying mix of down‑shift and up‑shift signals
    y = w_up * y_up + w_down * y_down

    # Simple normalisation to avoid clipping
    max_abs = np.max(np.abs(y))
    if max_abs > 1.0 and max_abs > 0.0:
        y = y / max_abs

    return y.astype(np.float32)

In [ ]:
try:
    vibrato
except NameError as e:
    raise NameError("Please define vibrato() before running this demo.") from e

# Try to reuse audio_in and fs from earlier cells; otherwise create a test tone
fs = int(globals().get('fs', 16000))
audio_g = globals().get('audio_in', None)

if audio_g is not None:
    x = np.asarray(audio_g, dtype=float)
    if x.ndim > 1:
        x = x.mean(axis=1)   # mono mix for processing
else:
    # fallback: synthetic test signal (mixture of two sinusoids)
    fs = 16000
    dur = 2.0
    t = np.arange(int(dur * fs)) / fs
    x = 0.6*np.sin(2*np.pi*220*t) + 0.3*np.sin(2*np.pi*440*t)

# Choose vibrato parameters
speed = 5.0      # vibrato rate in Hz
shift = 30.0     # maximum frequency shift in Hz

y_vib = vibrato(x, fs, speed_hz=speed, shift_hz=shift, square_curve=False)

# Short time window for waveform comparison
sec = 0.25
n = int(sec * fs)
t_short = np.arange(n) / fs

plt.figure(figsize=(10, 3))
plt.title(f'Vibrato effect (speed={speed:.1f} Hz, shift=±{shift:.0f} Hz)')
plt.plot(t_short, x[:n], label='original')
plt.plot(t_short, y_vib[:n], label='vibrato')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()

# Optional: spectrograms if you already defined show_spectrogram()
if 'show_spectrogram' in globals():
    show_spectrogram(x, fs, 'Original signal – spectrogram')
    show_spectrogram(y_vib, fs, 'Vibrato signal – spectrogram')

# Listen to the result
ipd.display(ipd.Audio(x, rate=fs))
ipd.display(ipd.Audio(y_vib, rate=fs))

## Time Delay Effect - Echo and Comb Filter

Many interesting voice FX can be achieved by delaying the signal and recombining it with itself.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T10:**
    
* Implement a function `echo()` which mixes a signal $s(t)$ with itself in a delayed version, i.e. $s(t-t_0)$. Experiment with various values for the delay $t_0$, and note the different effects you can achieve with delays
  * below $20$ msecs
  * between $20$ and $100$ msecs, and
  * above $100$ msecs.
</div>

In [ ]:
def echo(signal, fs, delay_ms, gain=0.5):
    """
    Simple echo: y[n] = x[n] + gain * x[n - D]

    Parameters
    ----------
    signal : array_like
        Input signal x[n].
    fs : int or float
        Sampling frequency in Hz.
    delay_ms : float
        Echo delay in milliseconds.
    gain : float, optional
        Amplitude of the delayed copy (0..1 is typical).

    Returns
    -------
    y : np.ndarray (float32)
        Output signal with echo, same length as input.
    """
    x = np.asarray(signal, dtype=float).squeeze()
    N = x.shape[0]
    if N == 0:
        return np.zeros(0, dtype=np.float32)

    # Number of delayed samples D = t0 * fs
    D = int(round(delay_ms * 1e-3 * fs))
    if D <= 0:
        return x.astype(np.float32)

    y = np.copy(x)

    if D < N:
        y[D:] += gain * x[:-D]

    # Simple normalization to prevent overflow
    m = np.max(np.abs(y))
    if m > 1.0 and m > 0:
        y = y / m

    return y.astype(np.float32)


In [ ]:
# Your code here to show an example of the echo effect
#
x = np.asarray(audio_in, dtype=float)
fs = int(fs)
if x.ndim > 1:
    x = x.mean(axis=1)

# Three different delays:
# < 20 ms
# 20–100 ms
# > 100 ms
y_10ms = echo(x, fs, delay_ms=10.0, gain=0.5)   # below 20 ms
y_50ms = echo(x, fs, delay_ms=50.0, gain=0.5)   # between 20 and 100 ms
y_200ms = echo(x, fs, delay_ms=200.0, gain=0.5)   # above 100 ms

# Draw a short waveform comparison (0.2 seconds)
sec = 0.2
n = int(sec * fs)
t_short = np.arange(n) / fs

plt.figure(figsize=(10, 6))

plt.subplot(3, 1, 1)
plt.title("Echo, delay = 10 ms (gain=0.5)")
plt.plot(t_short, x[:n], label="original")
plt.plot(t_short, y_10ms[:n], label="echo 10 ms", alpha=0.8)
plt.ylabel("Amplitude")
plt.legend()

plt.subplot(3, 1, 2)
plt.title("Echo, delay = 50 ms (gain=0.5)")
plt.plot(t_short, x[:n], label="original")
plt.plot(t_short, y_50ms[:n], label="echo 50 ms", alpha=0.8)
plt.ylabel("Amplitude")
plt.legend()

plt.subplot(3, 1, 3)
plt.title("Echo, delay = 200 ms (gain=0.5)")
plt.plot(t_short, x[:n], label="original")
plt.plot(t_short, y_200ms[:n], label="echo 200 ms", alpha=0.8)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.legend()

plt.tight_layout()
plt.show()

# Use SpecGram to examine changes in the spectrum/time domain.
LDFT = 1024
plt.figure(figsize=(11, 6))

plt.subplot(2, 2, 1)
plt.specgram(x, Fs=fs, NFFT=LDFT)
plt.title("Original")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

plt.subplot(2, 2, 2)
plt.specgram(y_10ms, Fs=fs, NFFT=LDFT)
plt.title("Echo 10 ms")

plt.subplot(2, 2, 3)
plt.specgram(y_50ms, Fs=fs, NFFT=LDFT)
plt.title("Echo 50 ms")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

plt.subplot(2, 2, 4)
plt.specgram(y_200ms, Fs=fs, NFFT=LDFT)
plt.title("Echo 200 ms")
plt.xlabel("Time (s)")

plt.tight_layout()
plt.show()

# Listening experience
ipd.display(ipd.Audio(x, rate=fs))
ipd.display(ipd.Audio(y_10ms, rate=fs))
ipd.display(ipd.Audio(y_50ms, rate=fs))
ipd.display(ipd.Audio(y_200ms, rate=fs))


## Comb Filtering

You should observe that with delays below $20$ msec in your function `echo()`, the signals combine to create a subtle ‘phasing’ effect. This is known as ‘comb filtering’ as the signal is effectively interfering with itself, and frequency components corresponding to multiples of the delay time are enhanced or cancelled out (due to ‘superposition’). Delays between $20$ and $100$ msecs give the effect of the voice being in a reverberant room. Delays above $100$ msecs sound like distant echoes.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T11:**
    
* Visualise the impulse response of your function `echo()` as well as the transfer function. Can you give an explanation from having a look at the transfer function, why this effect would be called *comb filter*?
</div>

In [ ]:
# Your code here
#
# ...
fs = 8000
delay_ms = 10.0                # Less than 20 ms will result in comb filtering
gain = 0.7                     # Echo Attenuation Coefficient

# Impulse response length
ir_len_s = 0.2
N = int(ir_len_s * fs)

# Structural Unit Impact δ[n]
x_imp = np.zeros(N)
x_imp[0] = 1.0

# Obtain the impulse response using echo()
h = echo(x_imp, fs, delay_ms, gain)

h = h[:N]

# Drawing of Impulse Response (Time Domain)
t = np.arange(len(h)) / fs * 1000  # millisecond
plt.figure(figsize=(8, 3))
plt.stem(t, h)
plt.xlabel('Time [ms]')
plt.ylabel('Amplitude')
plt.title(f'Impulse response of echo() (delay={delay_ms} ms, gain={gain})')
plt.tight_layout()
plt.show()


# Calculate the frequency response (transfer function)
N_fft = 4096
H = np.fft.fft(h, N_fft)
f = np.arange(N_fft) * fs / N_fft     # [Hz]

# Plot only the amplitude response from 0 to fs/2.
half = N_fft // 2
H_mag_db = 20 * np.log10(np.abs(H[:half]) + 1e-12)  # Add a small constant to avoid log(0).

plt.figure(figsize=(8, 3))
plt.plot(f[:half], H_mag_db)
plt.xlabel('Frequency [Hz]')
plt.ylabel('Magnitude [dB]')
plt.title('Transfer function |H(f)| of echo()')
plt.grid(True)
plt.tight_layout()
plt.show()



**Answer to question in Task T10:**

<span style="font-weight:bold;color:orange">
    ...In the echo() function, the output is the superposition of the original signal and a delayed signal.
When a unit impulse is input to the system, it produces several equally spaced impulses. Consequently, the frequency response exhibits a series of equally spaced peaks and troughs (peaks and “notches”) along the frequency axis. These periodic peaks and troughs resemble the teeth of a comb, hence the name comb filter. ...
</span>

## Flanger

It is possible to use an LFO to vary the delay. The resulting effect is known as a *flanger*.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T12:**
    
* Add an LFO to your ‘delay’ to create a ‘flanger’, and experiment with different settings. Note that you will need to scale the output of the LFO, and you will get different effects depending on whether the delayed signal is mixed with the original or not.
</div>

In [ ]:
def flanger(signal, fs,
            base_delay_ms=3.0,
            depth_ms=2.0,
            lfo_speed_hz=0.5,
            mix=0.7,
            square_curve=False):
    """
    Simple flanger based on a time‑varying delay.

    signal : Input signal
    fs : Sampling rate [Hz]
    base_delay_ms : Base Delay (ms)
    depth_ms : Range of delay variation caused by LFO (± ms)
    lfo_speed_hz : LFO Frequency (Hz)
    mix : Blend ratio between raw and delayed signals (0..1)
    """

    x = np.asarray(signal, dtype=float)
    if x.ndim > 1:
        x = x.mean(axis=1)   #  convert to mono
    N = x.shape[0]
    if N == 0:
        return np.zeros(0, dtype=np.float32)

    mix = float(np.clip(mix, 0.0, 1.0))

    # 1) Generate LFO: Output within the range [-1, +1]
    l = lfo(speed_hz=lfo_speed_hz,
            depth=1.0,              # get [-1, +1]
            num_samples=N,
            fs=fs,
            square_curve=square_curve)

    # Scale and pan the LFO output to create a time-varying delay (in milliseconds).
    # delay_ms[n] = base_delay_ms + depth_ms * l[n]
    delay_ms = base_delay_ms + depth_ms * l     # [base-depth, base+depth]
    delay_ms = np.clip(delay_ms, 0.0, None)     # Negative delay is not permitted.

    # Convert milliseconds to “samples”
    delay_samples = (delay_ms * fs / 1000.0).astype(int)

    y = np.zeros_like(x, dtype=float)

    for n in range(N):
        d = delay_samples[n]
        dry = x[n]
        if n - d >= 0:
            wet = x[n - d]
        else:
            wet = 0.0
        y[n] = (1.0 - mix) * dry + mix * wet

    m = np.max(np.abs(y))
    if m > 1.0 and m > 0.0:
        y = y / m

    return y.astype(np.float32)

In [ ]:
# Generate a 2-second 440 Hz sine wave
dur = 2.0
N = int(dur * fs)
t = np.arange(N) / fs
x_tone = np.sin(2 * np.pi * 440 * t)

y_tone = flanger(x_tone, fs,
          base_delay_ms=7.0,
          depth_ms=6.0,
          lfo_speed_hz=0.3,
          mix=0.95)

ipd.display(ipd.Audio(x_tone, rate=fs))   # Pure Tone
ipd.display(ipd.Audio(y_tone, rate=fs))   # Pure tone after flanger

# Frequency Analysis

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q5:**

* COM3502-4502-6502:
    * What does FFT stand for and what does an FFT do?
* COM4502-6502 ONLY: What is a DFT and how is it different from an FFT?
    
</div>

**Answer to Question Q5:**

<span style="font-weight:bold;color:orange">
    ...FFT stands for Fast Fourier Transform.
FFT is an efficient algorithm for computing the Discrete Fourier Transform (DFT) of a signal, that is, for converting a time-discrete signal in the time domain into its frequency-domain representation (containing frequency and amplitude/phase). ...
</span>

## Creating the Spectrogram Step-by-Step (Task T13 for COM4502-6502 only, Task T14 for all students)

The magnitude $|X[n, \ell]|$ of the STFT for all $n$ and $\ell$ is known as the [spectrogram](https://en.wikipedia.org/wiki/Spectrogram) of a signal. It is frequently used to analyse signals in the time-frequency domain, for instance by a [spectrum analyser](https://en.wikipedia.org/wiki/Spectrum_analyzer). It can be interpreted as a *image* of the signal with (block) time direction on the $x$ axis and (discrete) frequency $n$ on the y axis.

From Lab Sheet 3 we already know how to breack a long signal into block, a.k.a. frames.

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 13: Manual Spectrogram Calculation (for COM4502-6502 only)**
    
<ul>
    <li>
        Implement a function <code>calc_SpectralPoint(xk,n)</code> which calculates a spectral point for one discrete frequency $n$ from a input frame $x[k]$, i.e. a function which implements the well-known DFT equation
    $$
    \mathrm{DFT}\{x[k]\}   =  X[n] = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}
    $$
    for one fixed $n$.
    </li>
    <li>
        Implement a very similar function <code>calc_SpectralPointWindowed(xk,n)</code> which calculates a spectral point for one discrete frequency $n$ from a input frame $x[k]$, but in addition applies a window function $w[k]$ to the frame $x[k]$, i.e. the function should calculate
    $$
    \mathrm{DFT}\{w[k] \cdot x[k]\}   =  X^{\mathrm{w}}[n] = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} w[k] x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}
    $$
    for one fixed $n$. The window should have the same length $L_{\mathrm{DFT}}$ as your frame and should be one of the windows we discussed during the lecture.
    </li>
    <li>
        The functions above only calculate one spectral value at a time. To obtain a full spectrum, implement a function <code>calc_Manitude_Spectrum()</code> which transforms every windowed frame to the frequency domain and calculates all positive frequencies, i.e. for $0 \leq n \leq L_{\mathrm{DFT}}/2+1$.
    </li>
    <li>
        Create a function <code>create_spectrogram()</code>, which splits the complete input sequence (e.g. a loaded WAVE file) into blocks of length $L_{\mathrm{DFT}}$. These may be overlapping. For each block the spectrum should be calculated using the previously implemented function <code>calc_Manitude_Spectrum()</code> and all spectra should be collected to form a spectrogram (e.g. as columns of a matrix).
    </li>
    <li>
        Concatenate the resulting spectra to a spectrogram and display the resulting spectrogram. You can use <code>matplotlib</code>'s <code><a href="https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html">imshow()</a></code> function for manually plotting the spectrogram image. Note that a spectrogram is usually shown in dB scaling.
    </li>
    <li>
        Visualise the input signal $x[k]$ as a spectrogram for (i) a speech signal and (ii) for a chirp/sweep signal.
    </li>
</ul>
</div>

Implement the  DFT equation
    $$\mathrm{DFT}\{x[k]\}   =  X[n] = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}$$ for one fixed $n$:

In [ ]:
def calc_SpectralPoint(xk,n):
    '''
    Implementation of the Discrete Fourier Transform (DFT).
    Calculates the Fourier coefficient X[n] for one discrete frequency n

    Input:
    xk:     time domain signal vector
    n:      discrete frequency to be calculated

    Output
    Xn : discrete frequency domain point for frequency n
    '''

    # Your code here
    # ...

Implement DFT of windowed frame
$$\mathrm{DFT}\{w[k] \cdot x[k]\}   =  X^{\mathrm{w}}[n]  = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} w[k] x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}$$ for one fixed $n$.

In [ ]:
def calc_SpectralPointWindowed(xk,n,window=False):
    '''
    Implementation of the Discrete Fourier Transform (DFT).
    Calculates the Fourier coefficient X[n] for one discrete frequency n

    Input:
    xk:     time domain signal vector
    n:      discrete frequency to be calculated
    window: (optional): can be False (no window) or a window name from
            the list of available numpy window functions, e.g.
            np.hamming, np.bartlett, np.blackman, np.hanning, np.kaiser
            type: function
            (feel free to implement the window differently)

    Output
    Xn : dicrete frequency domain point for frequency n
    '''
    # Your code here
    # L_DFT = ???
    # ...

    if window == False:
        win = np.ones(L_DFT) # this is a window with no effect
    else:
        None # replace this by your own window

    # Your code here
    # ...

The following function <code>calc_Manitude_Spectrum()</code> is supposed to transform every (windowed or not windowed) frame to the frequency domain and to calculate all positive frequencies, i.e. $X[n]$ or $X^{\mathrm{w}}[n]$ for $0 \leq n \leq L_{\mathrm{DFT}}/2+1$.

In [ ]:
def calc_Manitude_Spectrum(xk):
    '''
    Compute Fourier coefficients up to the Nyquest Limit (fs/2), i.e. Xn for n=0,...,L_DFT/2
    using one of the two functions created before.
    and multiply the absolute value of the Fourier coefficients by 2,
    to account for the symmetry of the Fourier coefficients above the Nyquest Limit.
    '''
    # Your code here
    # ...

    # probably there should be a loop over n here

The following function <code>create_spectrogram()</code> should calculate all spectra needed for your spectrogram.

In [ ]:
def create_spectrogram(x, L_DFT=512, noverlap):
    '''
           x: original time series
       L_DFT: The number of data points used in each block for the DFT. The default value is 512.
    noverlap: The number of points of overlap between blocks. The default value is 256.
    '''
    # Your code here
    # ...



The following function can be used to actually display the created spectrogram.

In [ ]:
def plot_spectrogram( #...

The following code actually calculates and plots your spectrogram (for the two signals mentioned above). Feel free to adapt parameters `L_DFT` and `noverlap`

In [ ]:
# load or create signal

# create and plot spectrogram (generated using your functions above)
L_DFT    = 256 # DFT length
noverlap = 84  # number of overlapping samples
starts, spec = create_spectrogram( #...
plot_spectrogram(#...

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 14: Apply Short-Time Fourier Transform (STFT)**
    
<ul>
    <li>
        Explain the purpose of the Short-Time Fourier Transform (STFT) and how it differs from a regular Fourier Transform. Provide a brief(!) explanation below.
    </li>
    <li>
        Apply the STFT to the signal generated in Task T2 using a window size of $256$ samples with $50$% overlap (COM4502-6502 students should use the previously created code (if Task T13 was completed), existing functions can be used by COM3502 students (or if COM4502-6502 students did not complete Task T13).
    </li>
    <li>
        Interpret the spectrogram. What information does it provide about the signal? Briefly(!) explain what you see in the spectrogram and how it represents the frequency content over time.
    </li>
</ul>
</div>

In [ ]:
# ===== Task 14: STFT of the Task T2 signal =====
# It is assumed that the cell for Task2 has already been executed, where x represents three sine signals and fs = 8000 Hz.
x = x_T2
fs = fs_T2

# STFT
win_length = 256                 # Window length = 256 sampling points
hop = win_length // 2      # 50% overlap -> Step size 128
window = signal.windows.hann(win_length, sym=False)  # use Hann window

# Calculation STFT
f_stft, t_stft, Zxx = signal.stft(
    x,
    fs=fs,
    window=window,
    nperseg=win_length,
    noverlap=hop,
    nfft=win_length,
    boundary=None
)

# convert to dB plotting
eps = 1e-10
S_db = 20 * np.log10(np.abs(Zxx) + eps)

plt.figure(figsize=(8, 4))
plt.pcolormesh(t_stft, f_stft, S_db, shading="gouraud")
plt.title("STFT of Task T2 signal (256-sample window, 50% overlap)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.ylim(0, fs/2)   # only show 0 ~ Nyquist
cbar = plt.colorbar()
cbar.set_label("Magnitude (dB)")
plt.tight_layout()

**Answer to question in Task T14:**

<span style="font-weight:bold;color:orange">
    ...The Short-Time Fourier Transform (STFT) is employed to partition a lengthy signal into brief segments (utilising a window function),
    with a Fourier transform applied to each segment individually. This approach enables the observation of ‘how frequency components manifest at different points in time’.
    The standard Fourier transform applies a single transformation to the entire signal, revealing only the overall frequencies,
    yet fails to indicate when these frequencies occur temporally.
    Perfoming an STFT on the sine signals of task2 reveals three nearly horizontaal bright band consistently present in the spectrogarm, located at approximately 100Hz, 300Hz, 500Hz. These bands persist throughout the entire 2 second duration, indcationg that the signal is compose of these threew fixed-frequency sine waves over the entire time span. The varying brightness (cokour) reflects the differing amplitudes of the three components.....
</span>

## Pitch analysis

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 15: Pitch analysis**
    
<ul>
    <li>
         Extract and plot the pitch contour of a speech signal over time. Write code to estimate the pitch for each frame using an autocorrelation-based pitch detection algorithm. Briefly discuss the variations in pitch. What can pitch tell us about the speaker’s speech patterns? Provide a brief analysis of the pitch contour.
    </li>
</ul>
</div>

In [ ]:
# ===== Task 15: Pitch analysis using autocorrelation =====
# Use the raw audio input audio_in and sample rate audio_fs read in Task T1.

# Prepare the audio signal
x_pitch = np.asarray(audio_in, dtype=np.float32)
fs_pitch = int(audio_fs)
if x_pitch.ndim > 1:
    x_pitch = x_pitch.mean(axis=1)

# Frame parameters: 30 ms window length, 10 ms frame shift
frame_len = int(0.03 * fs_pitch)     # 30 ms
frame_shift = int(0.01 * fs_pitch)   # 10 ms
window = np.hamming(frame_len)       # Hamming windoww

num_frames = 1 + (len(x_pitch) - frame_len) // frame_shift

energies = np.zeros(num_frames)
for m in range(num_frames):
    start = m * frame_shift
    frame = x_pitch[start:start+frame_len] * window
    energies[m] = np.mean(frame**2)

energy_thresh = 0.1 * np.max(energies)   # Threshold = 10% of maximum energy

# Autocorrelation method estimates the fundamental frequency per frame
f_min = 50.0    # Minimum possible fundamental frequency (Hz)
f_max = 400.0   # Maximum possible fundamental frequency (Hz)

pitches_hz = np.zeros(num_frames)

frame_times = (np.arange(num_frames) * frame_shift + frame_len / 2) / fs_pitch

for m in range(num_frames):
    start = m * frame_shift
    frame = x_pitch[start:start+frame_len] * window

    if energies[m] < energy_thresh:
        pitches_hz[m] = 0.0
        continue

    acf_full = np.correlate(frame, frame, mode='full')
    acf = acf_full[len(frame)-1:]   # lag = 0,1,2,...

    lag_min = int(fs_pitch / f_max)
    lag_max = int(fs_pitch / f_min)
    if lag_max >= len(acf):
        lag_max = len(acf) - 1

    acf_segment = acf[lag_min:lag_max+1]
    best_rel = np.argmax(acf_segment)
    best_lag = lag_min + best_rel

    if acf[0] <= 0:
        pitches_hz[m] = 0.0
        continue

    peak_norm = acf[best_lag] / acf[0]
    if peak_norm < 0.3:
        pitches_hz[m] = 0.0
    else:
        pitches_hz[m] = fs_pitch / best_lag  # f0 = fs / lag

# Draw the fundamental frequency contour
plt.figure(figsize=(10, 3))
plt.plot(frame_times, pitches_hz, marker='o', linestyle='-')
plt.title("Pitch contour estimated with autocorrelation")
plt.xlabel("Time (s)")
plt.ylabel("Pitch (Hz)")
plt.ylim(0, 500)
plt.grid(True)
plt.tight_layout()
plt.show()


**Answer to question in Task T15:**

<span style="font-weight:bold;color:orange">
    ...your answer here ...
</span>

# Speech Synthesis

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 16: Synthesize a simple speech sound**
    
<ul>
    <li>
        Generate a vowel sound (e.g., “ah”) using a source-filter model, where a glottal pulse train with a frequency of 120 Hz is filtered by a vocal tract filter.
        Implement the glottal pulse train as a periodic signal and apply a simple formant-based filter. Plot the resulting waveform.
        <li>
            Hint: Use the following formant frequencies <code>formant_freqs = [730, 1090, 2440]</code> to create filters with bandwiths of <code>bandwidths = [80, 90, 120]</code>.
        </li>
    </li>
    <li>
        Plot the spectrogram of the synthesized sound. Compare it to the spectrogram of a real speech sample (which you can record yourself) containing the same vowel sound.
        Discuss any differences observed between synthetic and real speech.
    </li>
    <li>
        Briefly(!) discuss any differences observed between synthetic and real speech.
    </li>
</ul>
</div>

In [ ]:
fundamental_freq = 120             # Fundamental frequency in Hz for the glottal pulse
formant_freqs = [730, 1090, 2440]  # F1, F2, F3 for "ah" in Hz
bandwidths = [80, 90, 120]         # Bandwidths in Hz


# Your code here
# ...

**Answer to question in Task T16:**

<span style="font-weight:bold;color:orange">
    ...your answer here  (in differences observed between synthetic and real speech)...
</span>

# Equaliser (for COM4502-6502 only)

We want to design an equaliser like shown in the picture below as a hardware system.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/Yamaha_EQ-500_Graphic_Equalizer.jpg/1920px-Yamaha_EQ-500_Graphic_Equalizer.jpg" align="center" style="width: 500px;"/>
<center><span style="font-size:smaller">
    Picture taken from <a href="https://simple.wikipedia.org/wiki/Equalization_(audio)">Wikipedia</a>, license: <a href="https://creativecommons.org/licenses/by/2.0/">CC BY 2.0</a>
</span></center>


The following function realises one of the sliders in software.

In [ ]:
def peaking_filter(gain,center_freq,q,fs):
    """
    Derive coefficients for a peaking filter with a given amplitude and
     bandwidth.  All coefficients are calculated as described in Zölzer's
     DAFX book (ISBN: 0-471-49078-4, p. 50 - 55).  This algorithm assumes
     a constant q-term is used through the equation.

    Usage:     `b,a` = peaking_filter(gain,center_freq, q,fs)
                `gain` is the logarithmic gain (in dB)
                `center_freq` is the center frequency
                `q` is q-term equating to (Fb / Fc)
                `fs` is the sampling rate

    Author:    Jeff Tackett 08/22/05
    Port to Python by George Close 10/07/21
    """

    gain = np.float32(gain)
    k = np.tan((np.pi*center_freq)/fs)
    V0 = 10**((gain)/20)
    # invert gain if a cut
    if V0 < 1:
        V0 = 1/V0

    # Boost
    if gain > 0:
        b0 = (1 + ((V0/q)*k)+ k**2) / (1+((1/q)*k)+k**2)
        b1 = (2 * (k**2 - 1)) / (1 + ((1/q)*k) + k**2)
        b2 = (1 - ((V0/q)*k) + k**2) / (1 + ((1/q)*k) + k**2)
        a1 = b1
        a2 =  (1 - ((1/q)*k) + k**2) / (1 + ((1/q)*k) + k**2)
    # Cut
    elif gain <0:
        b0 = (1 + ((1/q)*k) + k**2) / (1 + ((V0/q)*k) + k**2)
        b1 =       (2 * (k**2 - 1)) / (1 + ((V0/q)*k) + k**2)
        b2 = (1 - ((1/q)*k) + k**2) / (1 + ((V0/q)*k) + k**2)
        a1 = b1
        a2 = (1 - ((V0/q)*k) + k**2) / (1 + ((V0/q)*k) + k**2)
    #gain is 0
    else:
        b0 = V0
        b1 = 0
        b2 = 0
        a1 = 0
        a2 = 0
    a = [  1, a1, a2]
    b = [ b0, b1, b2]
    return b,a

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T17: (for COM4502-6502 only)**
    
* Visualise the frequency response of one filter.
* Implement a cascade of filters to realise an equaliser.
* Visualise the frequency response of your equaliser filter and the input and (filtered) output signal.
    
</div>

In [ ]:
# Your code here
#
# ...

# Prepare for submission

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T18:**

* Clear all cell outputs to reduce the file size (in Jupyter Notebooks click on "Cell->All Output->Clear")
* Create a `.zip` file named `YourName.zip` containing this Jupyter Notebook files as well as all other files necessary to run this notebook (**if such exist**, e.g. if you created (additional) WAVE files).
* Hand in your `.zip` file via Blackboard.
    

<span style="font-weight:bold;color:red;text-align:center;">**Important: For marking, we expect your code to work ‘out of the box’.**</span> This means that no additional software should need to be installed to make the Notebook run. If you only used libraries known from the Speech Processing Lab classes, you should be safe here.
    
</div>